# Outfit generation

Loads the trained weights and the training embeddings, then recommends complete outfits for a given product.

**Method** (described in the report, implemented in `src/generation/`):
1. Draw random candidate outfits from products that never appear in a training outfit, following the class mix of the training outfits.
2. Encode each candidate to its 256-dimensional posterior mean.
3. Score each candidate by the Euclidean distance to its nearest training embedding.
4. For an input product, return the closest candidates that contain it.

Steps 2 and 3 are the expensive ones. `python -m scripts.generation.generate_outfits` runs them once for `num_candidates` outfits; this notebook runs a smaller version so each step is visible.

Distance to a known outfit is a hypothesis for compatibility, not a validated score.

In [ ]:
import os
import sys
from pathlib import Path

# Make `src` and `scripts` importable when the kernel starts in `notebooks/`.
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_processing.schema import PRODUCT_CLASSES
from src.utils import visualization as viz
from src.utils.setup_utilities import load_config

from scripts.generation.generate_outfits import load_catalog
from scripts.generation.recommend import recommend
from src.generation.candidates import configuration_probabilities, product_pools, sample_candidates_by_catalogue
from src.generation.embeddings import ProductImages, encode_outfits, load_training_embeddings
from src.generation.retrieval import nearest_training_outfit, rank_for_product
from src.models.vae256 import load_trained_model

In [ ]:
# Optional path to an alternative config file; defaults to `config.yml`.
CONFIG_PATH = os.environ.get("FASHION_CONFIG")

config = load_config(CONFIG_PATH)
generation = config["generation"]

N_CANDIDATES = 2000  # the script uses generation["num_candidates"]
POOL = generation["candidate_pool"]
TOP_K = generation["top_k"]
SEED = generation["seed"]

## Artifacts

In [ ]:
catalog, outfits = load_catalog(config)
exported = load_training_embeddings(config["training"]["model_data_path"])
training_embeddings = exported["latent_space"]
assert len(training_embeddings) == len(outfits), "embeddings and outfit table are out of sync"

model = load_trained_model(config["training"]["model_weights_path"])
images = ProductImages(
    catalog.set_index("cod_modelo_color")["des_filename"],
    config["data"]["image_width"],
    config["data"]["image_height"],
)
print(f"{len(catalog)} catalog products, {len(outfits)} training outfits, latent dimension {training_embeddings.shape[1]}")

## Candidate configurations

Candidates copy the configurations of the training outfits (which slots are filled) with the same frequencies. Products come from the selected pool: `unseen` means never in a training outfit.

In [ ]:
def short_name(slot):
    return slot.split(",")[0]


configurations = configuration_probabilities(outfits)
configurations["slots"] = configurations["pattern"].map(
    lambda pattern: " + ".join(short_name(slot) for slot, used in zip(PRODUCT_CLASSES, pattern) if used)
)
display(configurations[["slots", "count", "probability"]].head(8))

pools = product_pools(catalog, outfits, POOL)
pd.Series({short_name(slot): len(ids) for slot, ids in pools.items()}, name=f"{POOL} products").to_frame().T

## Sampling

Each candidate takes all its products from one catalogue (see the exploratory notebook), with the configuration frequencies of that catalogue's training outfits.

In [ ]:
candidates = sample_candidates_by_catalogue(catalog, outfits, POOL, N_CANDIDATES, SEED)
viz.show_outfits(candidates.sample(4, random_state=SEED), images, titles=["candidate"] * 4);

## Encoding

Each candidate is turned into a 128 × 960 strip exactly like a training outfit, including the JPEG round trip of the training archives, and encoded to its posterior mean.

In [ ]:
embeddings = encode_outfits(
    model,
    candidates,
    images,
    generation["encode_batch_size"],
    generation["match_training_jpeg"],
)
embeddings.shape

## Distance to the training outfits

In [ ]:
distances, nearest = nearest_training_outfit(embeddings, training_embeddings)
generated = candidates.assign(distance=distances, nearest_training_row=nearest)
generated["distance_percentile"] = generated["distance"].rank(pct=True)
print(generated["distance"].describe().round(3).to_string())
viz.plot_distance_histogram(distances);

### Closest and farthest candidates

Each candidate is shown above the training outfit it is closest to.

In [ ]:
def with_nearest_training_outfit(frame):
    rows, titles = [], []
    for _, row in frame.iterrows():
        rows.append(row[list(PRODUCT_CLASSES)])
        rows.append(outfits.iloc[int(row["nearest_training_row"])][list(PRODUCT_CLASSES)])
        titles += [f"candidate  d={row['distance']:.2f}", "nearest training"]
    return pd.DataFrame(rows), titles


pairs, titles = with_nearest_training_outfit(generated.nsmallest(3, "distance"))
viz.show_outfits(pairs, images, titles=titles);

In [ ]:
pairs, titles = with_nearest_training_outfit(generated.nlargest(3, "distance"))
viz.show_outfits(pairs, images, titles=titles);

## Recommendations for a product

Set `PRODUCT_ID` to a product present in the candidates, or leave it as `None` to pick one at random. The input product is framed in red in the outfits below.

In [ ]:
PRODUCT_ID = None

candidate_products = pd.unique(generated[list(PRODUCT_CLASSES)].stack())
product = PRODUCT_ID or np.random.default_rng(SEED).choice(candidate_products)
product_class = catalog.set_index("cod_modelo_color").loc[product, "des_product_class"]
occurrences = (generated[list(PRODUCT_CLASSES)] == product).any(axis=1).sum()
print(f"input product {product} ({product_class}), present in {occurrences} candidates")
plt.imshow(images(product))
plt.axis("off")
plt.show()

In [ ]:
best = rank_for_product(generated, product, TOP_K)
viz.show_outfits(
    best[list(PRODUCT_CLASSES)],
    images,
    highlight=product,
    titles=[f"#{rank}  d={distance:.2f}" for rank, distance in enumerate(best["distance"], start=1)],
);

In [ ]:
worst = rank_for_product(generated, product, TOP_K, worst=True)
viz.show_outfits(
    worst[list(PRODUCT_CLASSES)],
    images,
    highlight=product,
    titles=[f"worst #{rank}  d={distance:.2f}" for rank, distance in enumerate(worst["distance"], start=1)],
);

## Candidates generated around the product

Random candidates contain some products far more often than others. `recommend` instead generates candidates that all contain the input product, within its catalogue, and returns the closest ones (`python -m scripts.generation.recommend PRODUCT_ID` does the same from the command line).

In [ ]:
around, _ = recommend(product, TOP_K, num_candidates=1000, config_path=CONFIG_PATH)
viz.show_outfits(
    around[list(PRODUCT_CLASSES)],
    images,
    highlight=product,
    titles=[f"#{rank}  d={distance:.2f}" for rank, distance in enumerate(around["distance"], start=1)],
);

## Full run

`python -m scripts.generation.generate_outfits` writes the ranked table for all candidates. When it exists, the same query runs against it.

In [ ]:
generated_path = Path(generation["generated_outfits_path"])
if generated_path.exists():
    full = pd.read_csv(generated_path, dtype={slot: str for slot in PRODUCT_CLASSES})
    top = rank_for_product(full, product, TOP_K)
    print(f"{len(full)} generated outfits, {len(top)} shown")
    if len(top):
        viz.show_outfits(
            top[list(PRODUCT_CLASSES)],
            images,
            highlight=product,
            titles=[f"d={distance:.2f}" for distance in top["distance"]],
        )
else:
    print("Run scripts.generation.generate_outfits first.")